## Day 3 - Part 4: BERT, NLP의 판도를 바꾼 혁명가

### 과제 목표

1.  Hugging Face `datasets`와 `transformers` 라이브러리의 기본 사용법을 익힙니다.
2.  사전 학습된 BERT 모델을 특정 도메인(네이버 영화 리뷰)에 파인튜닝하는 전 과정을 직접 수행합니다.
3.  파인튜닝된 모델을 사용하여 새로운 텍스트의 감성을 분류하는 실용적인 예제를 완성합니다.

---

### 1. 환경 설정 및 데이터 로드

먼저 실습에 필요한 라이브러리를 설치하고, Hugging Face Hub에서 NSMC 데이터셋을 로드합니다.
NSMC 데이터셋은 `train`과 `test`로 나뉘어 있으며, 각각 'id', 'document'(리뷰), 'label'(0: 부정, 1: 긍정) 컬럼을 가집니다.

In [ ]:
# TODO: 아래 명령어를 실행하여 필요한 라이브러리를 설치하세요.
!pip install transformers datasets evaluate accelerate

import pandas as pd
import numpy as np
import plotly.express as px
from datasets import load_dataset

# TODO: `load_dataset` 함수를 사용하여 'nsmc' 데이터셋을 로드하세요.
dataset = load_dataset("nsmc")

print("전체 데이터셋 구조:")
print(dataset)

# TODO: 훈련 데이터셋을 pandas DataFrame으로 변환하고, `.head()`를 이용해 처음 5개 행을 출력하세요.
train_df = pd.DataFrame(dataset['train'])
print("\n훈련 데이터 샘플 (처음 5개):")
print(train_df.head())

### 2. 텍스트 전처리: 토크나이저(Tokenizer) 적용

BERT 모델에 텍스트를 입력하기 위해, 사전 학습된 모델에 맞는 토크나이저를 사용하여 텍스트를 숫자 시퀀스로 변환합니다.
여기서는 104개 언어를 지원하는 `bert-base-multilingual-cased` 모델과 그 토크나이저를 사용합니다.

In [ ]:
from transformers import AutoTokenizer

# TODO: 'bert-base-multilingual-cased' 체크포인트 이름을 변수에 저장하고, `AutoTokenizer.from_pretrained`로 토크나이저를 로드하세요.
model_checkpoint = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# TODO: 리뷰 텍스트('document' 컬럼)를 토큰화하는 함수를 정의하세요.
# HINT: `padding`은 'max_length'로, `truncation`은 True로, `max_length`는 128로 설정합니다.
def tokenize_function(examples):
    return tokenizer(examples["document"], padding="max_length", truncation=True, max_length=128)

# TODO: `dataset.map()` 함수를 사용하여 전체 데이터셋에 토큰화 함수를 적용하세요. `batched=True` 옵션을 사용하면 더 빠릅니다.
tokenized_datasets = dataset.map(tokenize_function, batched=True)

print("\n토큰화 후 데이터셋 샘플:")
print(tokenized_datasets['train'][0])

### 3. 모델 로드 및 훈련 준비

파인튜닝할 BERT 모델을 로드하고, `Trainer` API를 사용하기 위한 훈련 관련 설정을 정의합니다.

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate

# TODO: `AutoModelForSequenceClassification.from_pretrained`를 사용하여 모델을 로드하세요.
# 체크포인트는 위와 동일하게 사용하고, `num_labels`는 2로 설정합니다. (긍정/부정)
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2)

# TODO: `TrainingArguments`를 정의하여 훈련 하이퍼파라미터를 설정하세요.
# - output_dir: "./bert-nsmc-finetuned"
# - num_train_epochs: 2
# - learning_rate: 2e-5
# - evaluation_strategy: "epoch"
# - save_strategy: "epoch"
# - load_best_model_at_end: True
training_args = TrainingArguments(
    output_dir="./bert-nsmc-finetuned",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

# 평가 지표(accuracy)를 계산하는 함수
metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# TODO: `Trainer` 인스턴스를 생성하세요.
# model, args, train_dataset, eval_dataset, compute_metrics를 인자로 전달합니다.
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
)

### 4. 모델 파인튜닝 실행

이제 `trainer.train()`을 호출하여 파인튜닝을 시작합니다. GPU 환경에서 실행하는 것을 강력히 권장합니다. (Colab에서 런타임 유형을 GPU로 변경하세요) 훈련에는 수십 분에서 몇 시간까지 소요될 수 있습니다.

In [ ]:
# TODO: 아래 코드를 실행하여 모델 훈련을 시작하세요.
trainer.train()

### 5. 모델 평가 및 추론

훈련이 완료된 모델의 성능을 최종적으로 평가하고, 새로운 문장을 입력하여 감성 분석을 직접 테스트해 봅시다.

In [ ]:
import torch
from transformers import pipeline

# TODO: `trainer.evaluate()`를 호출하여 테스트셋에 대한 최종 성능을 확인하세요.
eval_results = trainer.evaluate()
print("\n최종 모델 평가 결과:")
print(eval_results)

# TODO: 훈련된 모델과 토크나이저를 사용하여 'sentiment-analysis' 파이프라인을 생성하세요.
sentiment_classifier = pipeline(
    "sentiment-analysis",
    model=trainer.model, # 훈련이 완료된 최적 모델 사용
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

# TODO: 자신만의 영화 리뷰 문장을 만들어 모델의 예측을 테스트해보세요.
my_review = "기대 안하고 봤는데 생각보다 재밌었어요. 킬링타임용으로 추천합니다."

result = sentiment_classifier(my_review)
label = "긍정" if result[0]['label'] == 'LABEL_1' else "부정"

print(f'\n리뷰: "{my_review}"')
print(f'예측 결과: {label} (신뢰도: {result[0]["score"]:.4f})')